In [2]:
# Robust, T4-friendly training script (single-stage 224x224 Vit-Base)
# Keep dataset paths unchanged.
# Requirements: torch, torchvision, timm, scikit-learn, tqdm, matplotlib, seaborn

import os, time, math, random, zipfile
from collections import OrderedDict
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel
from torchvision import transforms, datasets
import timm
from tqdm import tqdm

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from torch.utils.data.sampler import WeightedRandomSampler

# ----------------------------- USER CONFIG -----------------------------
DRIVE_MOUNT = True
ZIP_PATH = "/content/drive/MyDrive/d.zip"   # optional zipped dataset
EXTRACT_PATH = "/content/dataset"
TRAIN_DIR = "/content/dataset/d/train"
VAL_DIR   = "/content/dataset/d/val"
GRAPH_DIR = "/content/drive/MyDrive/results_4k"
CHECKPOINT_PATH = "/content/drive/MyDrive/best_model_t4.pt"
FINAL_MODEL_PATH = "/content/drive/MyDrive/final_model_t4.pt"

# T4-friendly defaults
PREFERRED_MODEL = "vit_base_patch16_224"   # smaller than vit_large
FALLBACK_MODEL  = "efficientnet_b3"

IMG_SIZE = 224   # single-stage, match pretrained ViT size to avoid pos-embed issues

REQUESTED_NUM_WORKERS = 2
BATCH_SIZE = 12
NUM_WORKERS = min(REQUESTED_NUM_WORKERS, max(1, (os.cpu_count() or 2)//2))
SEED = 42
PHASE1_EPOCHS = 4    # head-only
PHASE2_EPOCHS = 18   # finetune
N_UNFREEZE = 8
HEAD_LR = 4e-4
FINETUNE_LR = 2e-5
WEIGHT_DECAY = 1e-2
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
CUTMIX_PROB = 0.4
MAX_GRAD_NORM = 1.0
SWA_FINISH_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 8

# LLRD
LAYER_DECAY = 0.8

# Lookahead params
LOOKAHEAD_K = 5
LOOKAHEAD_ALPHA = 0.5

# TTA transforms for eval
TTA_TRANSFORMS = [
    lambda x: x,
    lambda x: torch.flip(x, dims=[-1]),
    lambda x: torch.rot90(x, k=1, dims=[-2, -1]),
]

torch.backends.cudnn.benchmark = True
# -----------------------------------------------------------------------

# reproducibility
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# mount Drive if Colab
if DRIVE_MOUNT:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass

# extract ZIP if provided
if os.path.exists(ZIP_PATH) and not os.path.exists(EXTRACT_PATH):
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(EXTRACT_PATH)
            print("Extracted dataset to", EXTRACT_PATH)
    except Exception as e:
        print("Zip extraction failed:", e)
else:
    print("No zip to extract or already available.")

os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| NUM_WORKERS:", NUM_WORKERS)

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(VAL_DIR):
    raise FileNotFoundError(f"Train/Val directories not found. Expected: {TRAIN_DIR} and {VAL_DIR}")

# classes & counts
classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
num_classes = len(classes)
print(f"Detected {num_classes} classes: {classes}")

# -------------------- transforms & dataloaders --------------------
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.06)),
        transforms.RandomResizedCrop(img_size, scale=(0.6,1.0), ratio=(0.9,1.11)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.18,0.12,0.12,0.02),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.15, scale=(0.01,0.12))
    ])
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.06)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
    return train_tf, val_tf

def make_dataloaders(img_size, batch_size=BATCH_SIZE):
    train_tf, val_tf = get_transforms(img_size)
    train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
    val_ds = datasets.ImageFolder(VAL_DIR, transform=val_tf)
    y_train = [lbl for _, lbl in train_ds.samples]
    if num_classes > 1 and len(y_train)>0:
        cw_np = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
        class_weights = torch.from_numpy(cw_np).float().to(device)
        sample_weights = np.array([cw_np[y] for y in y_train], dtype=np.float32)
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    else:
        class_weights = None
        sampler = None
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size,
                                               sampler=sampler if sampler is not None else None,
                                               shuffle=(sampler is None),
                                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                                             num_workers=NUM_WORKERS, pin_memory=True)
    return train_ds, val_ds, train_loader, val_loader, class_weights

# -------------------- model + safe pos-embed loader --------------------
def interpolate_pos_embed_from_state(src_pos_tensor, model, target_img_size, verbose=False):
    pos_src = src_pos_tensor.detach().cpu()
    if pos_src.ndim != 3:
        raise ValueError("src_pos_tensor must be shape (1, 1+N, C)")
    cls_tok = pos_src[:, :1, :]
    grid_tok = pos_src[:, 1:, :]
    n_old = grid_tok.shape[1]; old_grid = int(math.sqrt(n_old))
    if old_grid * old_grid != n_old:
        raise ValueError("src pos grid not square; can't interpolate")
    # find patch_size
    patch_size = None
    if hasattr(model, "patch_embed"):
        pe = model.patch_embed
        if hasattr(pe, "patch_size"):
            ps = pe.patch_size; patch_size = ps[0] if isinstance(ps, (list,tuple)) else ps
        else:
            try:
                ks = pe.proj.kernel_size; patch_size = ks[0] if isinstance(ks,(list,tuple)) else ks
            except Exception:
                patch_size = None
    if patch_size is None:
        raise RuntimeError("Cannot determine patch_size for interpolation")
    new_grid = target_img_size // patch_size
    if new_grid * new_grid == n_old:
        if verbose: print("pos_embed grid matches; no interp.")
        return pos_src
    grid_tok = grid_tok.reshape(1, old_grid, old_grid, pos_src.shape[-1]).permute(0,3,1,2)
    grid_resized = F.interpolate(grid_tok, size=(new_grid, new_grid), mode='bicubic', align_corners=False)
    grid_resized = grid_resized.permute(0,2,3,1).reshape(1, new_grid*new_grid, pos_src.shape[-1])
    new_pos = torch.cat([cls_tok, grid_resized], dim=1)
    if verbose:
        print(f"Interpolated pos_embed {old_grid}x{old_grid} -> {new_grid}x{new_grid} (patch {patch_size})")
    return new_pos

def create_model_with_pretrained(primary, fallback, target_img_size, pretrained_src_img_size=224, verbose=False):
    pretrained_sd = None
    try:
        src_model = timm.create_model(primary, pretrained=True, num_classes=num_classes, img_size=pretrained_src_img_size)
        pretrained_sd = src_model.state_dict()
        if verbose: print(f"Loaded pretrained {primary} (src size {pretrained_src_img_size})")
    except Exception as e:
        if verbose: print("Could not load pretrained source:", e)
        pretrained_sd = None
    # create target model
    try:
        model = timm.create_model(primary, pretrained=False, num_classes=num_classes, img_size=target_img_size)
        used = primary
    except Exception:
        model = timm.create_model(fallback, pretrained=False, num_classes=num_classes, img_size=target_img_size)
        used = fallback
    # copy compatible weights + interpolate pos_embed when present
    if pretrained_sd is not None:
        sd_to_load = {}
        for k, v in pretrained_sd.items():
            if k == "pos_embed" or k.endswith(".pos_embed"):
                try:
                    new_pos = interpolate_pos_embed_from_state(v, model, target_img_size, verbose=verbose)
                    sd_to_load[k] = new_pos.to(model.pos_embed.dtype)
                except Exception:
                    continue
            else:
                if k in model.state_dict() and model.state_dict()[k].shape == v.shape:
                    sd_to_load[k] = v
        model.load_state_dict(sd_to_load, strict=False)
        if verbose: print("Loaded pretrained weights into model (safe keys + pos_interp).")
    return model, used

# -------------------- loss, optimizer helpers --------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma; self.weight = weight; self.reduction = reduction
    def forward(self, logits, target):
        logpt = F.log_softmax(logits, dim=1)
        pt = torch.exp(logpt)
        logpt = logpt.gather(1, target.unsqueeze(1)).squeeze(1)
        pt = pt.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = - (1 - pt) ** self.gamma * logpt
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean() if self.reduction == 'mean' else loss.sum()

def get_layer_id_for_vit(name, num_layers):
    if name.startswith("patch_embed") or name.startswith("pos_embed"): return 0
    if "blocks." in name:
        try:
            part = name.split("blocks.")[1]; idx = int(part.split(".")[0]); return idx + 1
        except Exception: return num_layers
    if name.startswith("cls_token") or "norm" in name or "head" in name or "classifier" in name:
        return num_layers + 1
    return num_layers

def param_groups_lrrd(model, base_lr, layer_decay):
    groups = {}
    if hasattr(model, "blocks"):
        num_layers = len(model.blocks) + 2
        for name, p in model.named_parameters():
            if not p.requires_grad: continue
            layer_id = get_layer_id_for_vit(name, num_layers)
            scale = layer_decay ** (num_layers - layer_id)
            lr = float(base_lr * scale)
            key = f"lr_{lr:.10f}"
            groups.setdefault(key, {"params": [], "lr": lr})
            groups[key]["params"].append(p)
    else:
        groups["all"] = {"params":[p for p in model.parameters() if p.requires_grad], "lr": base_lr}
    return list(groups.values())

class Lookahead:
    def __init__(self, optimizer, k=LOOKAHEAD_K, alpha=LOOKAHEAD_ALPHA):
        self.optimizer = optimizer; self.k = k; self.alpha = alpha; self._step = 0
        self.slow_weights = []
        for group in optimizer.param_groups:
            grp = []
            for p in group['params']:
                sw = p.detach().clone(); sw.requires_grad_(False); grp.append(sw)
            self.slow_weights.append(grp)
    def zero_grad(self): self.optimizer.zero_grad()
    def step(self):
        self._step += 1
        if self._step % self.k == 0:
            for g_idx, group in enumerate(self.optimizer.param_groups):
                for p_idx, p in enumerate(group['params']):
                    if p.grad is None: continue
                    slow = self.slow_weights[g_idx][p_idx]
                    slow.data.add_(self.alpha, p.data - slow.data)
                    p.data.copy_(slow.data)
    @property
    def param_groups(self): return self.optimizer.param_groups

def mixup_data(x, y, alpha=MIXUP_ALPHA):
    if alpha <= 0: return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    if x.size(0) == 1: return x, y, None, 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def cutmix_data(x, y, alpha=CUTMIX_ALPHA):
    if alpha <= 0: return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    b, c, h, w = x.size()
    if b == 1: return x, y, None, 1.0
    idx = torch.randperm(b).to(x.device)
    cut_rat = math.sqrt(1. - lam)
    cut_w = int(w * cut_rat); cut_h = int(h * cut_rat)
    cx = np.random.randint(w); cy = np.random.randint(h)
    x1 = np.clip(cx - cut_w // 2, 0, w); y1 = np.clip(cy - cut_h // 2, 0, h)
    x2 = np.clip(cx + cut_w // 2, 0, w); y2 = np.clip(cy + cut_h // 2, 0, h)
    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_adj = 1.0 - ((x2 - x1) * (y2 - y1) / float(w * h))
    return x, y, y[idx], lam_adj

def mixup_criterion(preds, y_a, y_b, lam, loss_fn):
    return lam * loss_fn(preds, y_a) + (1 - lam) * loss_fn(preds, y_b)

# -------------------- metrics & TTA --------------------
def compute_metrics_all(y_true_list, y_pred_list, y_prob_array, class_names):
    y_true = np.array(y_true_list); y_pred = np.array(y_pred_list)
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    n = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n)))
    if y_true_bin.ndim == 1: y_true_bin = y_true_bin.reshape(-1, 1)
    if y_true_bin.shape[1] == 1: y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
    aucs = []; roc_auc = {}; missing=[]
    for i in range(n):
        if np.sum(y_true_bin[:, i]) == 0:
            missing.append(class_names[i]); continue
        try:
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob_array[:, i])
            roc_auc[i] = auc(fpr, tpr); aucs.append(roc_auc[i])
        except Exception:
            missing.append(class_names[i])
    macro_auc = float(np.mean(aucs)) if len(aucs)>0 else 0.0
    return acc, report, roc_auc, None, None, macro_auc, missing

def tta_predict_batch(model, imgs, tta_transforms=TTA_TRANSFORMS):
    model.eval(); probs_sum = None
    with torch.no_grad():
        for tfn in tta_transforms:
            aug = tfn(imgs)
            logits = model(aug)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(tta_transforms)

# -------------------- training/validation --------------------
scaler = torch.amp.GradScaler()

def human_gpu_mem():
    if not torch.cuda.is_available(): return ""
    return f"{torch.cuda.memory_reserved(device)/1024**3:.2f}GB"

def train_one_epoch(model, loader, optimizer_wrapper, loss_fn, epoch, total_epochs, scheduler=None):
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    pbar = tqdm(enumerate(loader, start=1), total=len(loader),
                desc=f"Train {epoch}/{total_epochs}", ncols=120, leave=False)
    for batch_idx, (imgs, labels) in pbar:
        imgs = imgs.to(device); labels = labels.to(device)
        if np.random.rand() < CUTMIX_PROB:
            imgs_m, y_a, y_b, lam = cutmix_data(imgs.clone(), labels)
        else:
            imgs_m, y_a, y_b, lam = mixup_data(imgs, labels)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(imgs_m)
            if y_b is None:
                loss = loss_fn(logits, labels)
            else:
                loss = mixup_criterion(logits, y_a, y_b, lam, loss_fn)
        base_opt = optimizer_wrapper.optimizer if hasattr(optimizer_wrapper, "optimizer") else optimizer_wrapper
        scaler.scale(loss).backward()
        if MAX_GRAD_NORM is not None:
            scaler.unscale_(base_opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        try:
            scaler.step(base_opt)
        except Exception as e:
            print("Optimizer step failed:", e)
            scaler.update()
            base_opt.zero_grad()
            continue
        scaler.update()
        # Lookahead slow-sync
        if hasattr(optimizer_wrapper, "step") and hasattr(optimizer_wrapper, "optimizer"):
            optimizer_wrapper.step()
            optimizer_wrapper.zero_grad()
        else:
            base_opt.zero_grad()
        if scheduler is not None:
            try:
                scheduler.step()
            except Exception:
                pass
        running_loss += float(loss.item()) * imgs.size(0)
        preds = logits.detach().argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        running_epoch_loss = running_loss / total if total>0 else running_loss
        running_epoch_acc = correct / total if total>0 else 0.0
        postfix = {"b_loss": f"{loss.item():.4f}", "e_loss": f"{running_epoch_loss:.4f}", "e_acc": f"{running_epoch_acc:.4f}"}
        if torch.cuda.is_available(): postfix["GPU"] = human_gpu_mem()
        pbar.set_postfix(postfix, refresh=False)
    pbar.close()
    avg_loss = running_loss / len(loader.dataset) if len(loader.dataset)>0 else 0.0
    return avg_loss, correct / total if total>0 else 0.0

def validate(model, loader, loss_fn, epoch=None):
    model.eval()
    running_loss = 0.0; all_preds=[]; all_labels=[]; all_probs=[]
    pbar = tqdm(enumerate(loader, start=1), total=len(loader), desc=f"Val {epoch}" if epoch else "Val", ncols=120, leave=False)
    with torch.no_grad():
        for batch_idx, (imgs, labels) in pbar:
            imgs = imgs.to(device); labels = labels.to(device)
            probs = tta_predict_batch(model, imgs)
            preds = np.argmax(probs, axis=1)
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                logits = model(imgs)
                loss = loss_fn(logits, labels)
            running_loss += float(loss.item()) * imgs.size(0)
            all_probs.append(probs); all_preds.extend(list(preds)); all_labels.extend(list(labels.cpu().numpy()))
            batch_acc = (preds == labels.cpu().numpy()).mean()
            running_epoch_loss = running_loss / (len(all_labels) if len(all_labels)>0 else 1)
            running_epoch_acc = accuracy_score(all_labels, all_preds) if len(all_labels)>0 else 0.0
            postfix = {"b_loss": f"{loss.item():.4f}", "b_acc": f"{batch_acc:.3f}", "e_loss": f"{running_epoch_loss:.4f}", "e_acc": f"{running_epoch_acc:.4f}"}
            if torch.cuda.is_available(): postfix["GPU"] = human_gpu_mem()
            pbar.set_postfix(postfix, refresh=False)
    pbar.close()
    val_loss = running_loss / len(loader.dataset) if len(loader.dataset)>0 else 0.0
    all_probs = np.vstack(all_probs) if all_probs else np.zeros((0, num_classes))
    return val_loss, all_labels, all_preds, all_probs

# -------------------- orchestration --------------------
history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[], "val_auc_macro":[]}
best_val_acc = -1.0; best_score = -1e9
total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS

# dataloaders
train_ds, val_ds, train_loader, val_loader, class_weights = make_dataloaders(IMG_SIZE, batch_size=BATCH_SIZE)
print("Train samples:", len(train_ds), "Val samples:", len(val_ds))

# create model (target img size = pretrained size => no pos embed mismatch)
print("Creating model (this may download pretrained weights once)...")
model, used_model = create_model_with_pretrained(PREFERRED_MODEL, FALLBACK_MODEL, target_img_size=IMG_SIZE, pretrained_src_img_size=IMG_SIZE, verbose=True)
model.to(device)

# freeze then unfreeze head
for n, p in model.named_parameters(): p.requires_grad = False
unfreeze_cnt = 0
for name, p in model.named_parameters():
    if ("head" in name) or ("classifier" in name) or name.endswith("fc") or "proj" in name or "norm" in name:
        p.requires_grad = True; unfreeze_cnt += 1
if unfreeze_cnt == 0:
    # fallback: unfreeze last few params
    for name, p in list(model.named_parameters())[-80:]:
        p.requires_grad = True
print("Unfrozen head params approx:", unfreeze_cnt)

# loss
if class_weights is not None:
    loss_fn = FocalLoss(gamma=2.0, weight=class_weights)
else:
    loss_fn = FocalLoss(gamma=2.0, weight=None)

# optimizer for head-only phase
param_groups = param_groups_lrrd(model, HEAD_LR, LAYER_DECAY)
opt_base = AdamW(param_groups, lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
optimizer = Lookahead(opt_base, k=LOOKAHEAD_K, alpha=LOOKAHEAD_ALPHA)
scheduler = CosineAnnealingLR(opt_base, T_max=max(1, PHASE1_EPOCHS * max(1, len(train_loader))), eta_min=1e-7)

swa_model = AveragedModel(model)
use_swa = True
swa_start = PHASE1_EPOCHS + max(1, PHASE2_EPOCHS - SWA_FINISH_EPOCHS)

# Phase 1: head-only
stopping_counter = 0
for epoch in range(1, PHASE1_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, epoch, total_epochs, scheduler=scheduler)
    val_loss, y_true, y_pred, y_prob = validate(model, val_loader, loss_fn, epoch)
    acc, report, _, _, _, val_auc_macro, _ = compute_metrics_all(y_true, y_pred, y_prob, classes)
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss); history["val_acc"].append(acc); history["val_auc_macro"].append(val_auc_macro)
    print(f"Phase1 E{epoch}: tr_acc={tr_acc:.4f} val_acc={acc:.4f} val_auc={val_auc_macro:.4f} val_loss={val_loss:.4f} time={(time.time()-t0):.1f}s")
    combined = acc + val_auc_macro - val_loss*0.01
    if acc > best_val_acc:
        best_val_acc = acc; best_score = combined
        torch.save(model.state_dict(), CHECKPOINT_PATH); stopping_counter = 0; print("Saved best (phase1).")
    else:
        stopping_counter += 1; print(f"No improvement {stopping_counter}/{EARLY_STOPPING_PATIENCE}")
    if stopping_counter >= EARLY_STOPPING_PATIENCE:
        print("Early stopping Phase1"); break

# Phase 2: Unfreeze and finetune
print("Unfreezing last blocks for finetune")
try:
    if hasattr(model, "blocks"):
        total_blocks = len(model.blocks)
        for i in range(max(0, total_blocks - N_UNFREEZE), total_blocks):
            for p in model.blocks[i].parameters(): p.requires_grad = True
    elif hasattr(model, "transformer") and hasattr(model.transformer, "blocks"):
        total_blocks = len(model.transformer.blocks)
        for i in range(max(0, total_blocks - N_UNFREEZE), total_blocks):
            for p in model.transformer.blocks[i].parameters(): p.requires_grad = True
    else:
        # fallback: unfreeze many last params
        ct = 0
        for name, p in reversed(list(model.named_parameters())):
            p.requires_grad = True; ct += 1
            if ct > 500: break
except Exception:
    for p in model.parameters(): p.requires_grad = True

# optimizer for finetune
param_groups = param_groups_lrrd(model, FINETUNE_LR, LAYER_DECAY)
opt_base = AdamW(param_groups, lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)
optimizer = Lookahead(opt_base, k=LOOKAHEAD_K, alpha=LOOKAHEAD_ALPHA)
scheduler = CosineAnnealingLR(opt_base, T_max=max(1, PHASE2_EPOCHS * max(1, len(train_loader))), eta_min=1e-7)

# update loss_fn if class weights changed
if class_weights is not None:
    loss_fn = FocalLoss(gamma=2.0, weight=class_weights)
else:
    loss_fn = FocalLoss(gamma=2.0, weight=None)

ema_decay = 0.999
ema_params = [p.detach().clone().to(device) for p in model.parameters()]

stopping_counter = 0
for epoch in range(PHASE1_EPOCHS+1, PHASE1_EPOCHS+PHASE2_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, epoch, total_epochs, scheduler=scheduler)
    with torch.no_grad():
        for i, p in enumerate(model.parameters()):
            ema_params[i].mul_(ema_decay).add_(p.detach() * (1.0 - ema_decay))
    val_loss, y_true, y_pred, y_prob = validate(model, val_loader, loss_fn, epoch)
    acc, report, _, _, _, val_auc_macro, _ = compute_metrics_all(y_true, y_pred, y_prob, classes)
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss); history["val_acc"].append(acc); history["val_auc_macro"].append(val_auc_macro)
    if use_swa and epoch >= swa_start: swa_model.update_parameters(model)
    print(f"Phase2 E{epoch}: tr_acc={tr_acc:.4f} val_acc={acc:.4f} val_auc={val_auc_macro:.4f} val_loss={val_loss:.4f} time={(time.time()-t0):.1f}s")
    combined = acc + val_auc_macro - val_loss*0.01
    if acc > best_val_acc:
        best_val_acc = acc; best_score = combined
        torch.save(model.state_dict(), CHECKPOINT_PATH); stopping_counter = 0; print("Saved best (phase2).")
    else:
        stopping_counter += 1; print(f"No improvement {stopping_counter}/{EARLY_STOPPING_PATIENCE}")
    if stopping_counter >= EARLY_STOPPING_PATIENCE:
        print("Early stopping Phase2"); break

# finalize SWA
final_model = model
if use_swa:
    try:
        torch.optim.swa_utils.update_bn(train_loader, swa_model)
        final_model = swa_model.module if hasattr(swa_model, "module") else swa_model
    except Exception as e:
        print("SWA update_bn failed:", e); final_model = model

# load best ckpt
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    try:
        final_model.load_state_dict(ckpt)
    except Exception:
        final_model.load_state_dict(ckpt, strict=False)

final_model.to(device); final_model.eval()

# build EMA model
try:
    ema_model = timm.create_model(used_model, pretrained=False, num_classes=num_classes, img_size=IMG_SIZE).to(device)
    ema_model.load_state_dict(final_model.state_dict(), strict=False)
    with torch.no_grad():
        for p_target, p_ema in zip(ema_model.parameters(), ema_params):
            p_target.copy_(p_ema)
except Exception:
    ema_model = final_model

# evaluate
val_loss, y_true, y_pred, y_prob = validate(final_model, val_loader, loss_fn, epoch="final")
acc, report, roc_auc_dict, fpr, tpr, macro_auc, missing = compute_metrics_all(y_true, y_pred, y_prob, classes)
print(f"\nFINAL VAL ACC (final_model) = {acc*100:.2f}%\nMacro AUC = {macro_auc:.4f}")
print(report)
if missing: print("Skipped ROC for classes:", missing)

val_loss_e, y_true_e, y_pred_e, y_prob_e = validate(ema_model, val_loader, loss_fn, epoch="ema")
acc_e, report_e, _, _, _, macro_auc_e, missing_e = compute_metrics_all(y_true_e, y_pred_e, y_prob_e, classes)
print(f"\nFINAL VAL ACC (EMA model) = {acc_e*100:.2f}%\nMacro AUC (EMA) = {macro_auc_e:.4f}")

selected_model = ema_model if acc_e > acc else final_model
selected_acc = acc_e if acc_e > acc else acc
selected_report = report_e if acc_e > acc else report

torch.save(selected_model.state_dict(), FINAL_MODEL_PATH)
print(f"\nSaved final model to {FINAL_MODEL_PATH} (val_acc {selected_acc*100:.2f}%)")
print(selected_report)

# save plots
os.makedirs(GRAPH_DIR, exist_ok=True)
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(8,5))
plt.plot(epochs_range, history["train_acc"], marker='o', label='Train Acc')
plt.plot(epochs_range, history["val_acc"], marker='o', label='Val Acc')
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Accuracy over Epochs"); plt.grid(True, linestyle='--', alpha=0.4); plt.legend()
plt.savefig(os.path.join(GRAPH_DIR, "accuracy.png"), dpi=150); plt.close()
plt.figure(figsize=(8,5))
plt.plot(epochs_range, history["train_loss"], marker='o', label='Train Loss')
plt.plot(epochs_range, history["val_loss"], marker='o', label='Val Loss')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss over Epochs"); plt.grid(True, linestyle='--', alpha=0.4); plt.legend()
plt.savefig(os.path.join(GRAPH_DIR, "loss.png"), dpi=150); plt.close()
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
plt.figure(figsize=(7,6)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.tight_layout()
plt.savefig(os.path.join(GRAPH_DIR, "confusion_matrix.png"), dpi=150); plt.close()
print("Plots saved to:", GRAPH_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No zip to extract or already available.
Device: cuda | NUM_WORKERS: 1
Detected 2 classes: ['benign', 'malignant']
Train samples: 2637 Val samples: 660
Creating model (this may download pretrained weights once)...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loaded pretrained vit_base_patch16_224 (src size 224)
pos_embed grid matches; no interp.
Loaded pretrained weights into model (safe keys + pos_interp).
Unfrozen head params approx: 78


Train 1/22:   0%|                                                                               | 0/220 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Train 1/22:   1%|▏              | 3/220 [00:01<01:56,  1.86it/s, b_loss=0.2341, e_loss=0.5185, e_acc=0.4444, GPU=2.78GB]/tmp/ipython-input-4165719238.py:281: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha = 1) (Triggered internally at /pytorch/torch/csrc/utils/python_arg

Phase1 E1: tr_acc=0.6906 val_acc=0.8727 val_auc=0.9557 val_loss=0.0712 time=64.7s
Saved best (phase1).


Phase1 E2: tr_acc=0.7254 val_acc=0.8788 val_auc=0.9615 val_loss=0.0693 time=65.8s
Saved best (phase1).


... (17 outputs left out)

Phase2 E20: tr_acc=0.8013 val_acc=0.9121 val_auc=0.9719 val_loss=0.0635 time=65.3s
No improvement 7/8


Phase2 E21: tr_acc=0.7728 val_acc=0.9076 val_auc=0.9717 val_loss=0.0639 time=64.1s
No improvement 8/8
Early stopping Phase2



FINAL VAL ACC (final_model) = 91.52%
Macro AUC = 0.9716
              precision    recall  f1-score   support

      benign       0.94      0.91      0.92       360
   malignant       0.89      0.93      0.91       300

    accuracy                           0.92       660
   macro avg       0.91      0.92      0.91       660
weighted avg       0.92      0.92      0.92       660




FINAL VAL ACC (EMA model) = 89.85%
Macro AUC (EMA) = 0.9654

Saved final model to /content/drive/MyDrive/final_model_t4.pt (val_acc 91.52%)
              precision    recall  f1-score   support

      benign       0.94      0.91      0.92       360
   malignant       0.89      0.93      0.91       300

    accuracy                           0.92       660
   macro avg       0.91      0.92      0.91       660
weighted avg       0.92      0.92      0.92       660

Plots saved to: /content/drive/MyDrive/results_4k
